<a href="https://colab.research.google.com/github/gopalstud86/GenAI-Assignments/blob/main/Bronze%20Badge%20Assignments/Problem5_RAG%2BLLM%20with%20Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from huggingface_hub import login
from langchain.prompts import PromptTemplate

#Use below code in login before running this code, HF should be hf
#HF_VWEqhXViefrePmPZXDOqxEllmqvCHOFhRj
login(token="")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = Ollama(model="llama3")

#pdf_file = "https://raw.githubusercontent.com/gopalstud86/GenAI-Assignments/refs/heads/main/Bronze%20Badge%20Assignments/Datasets/Policy.pdf"

pdf_file = ("Docs/Policy.pdf")
# Load PDF
loader = PyPDFLoader(pdf_file)
documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", "", "-"]
)
docs = splitter.split_documents(documents)

# Define prompt template
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are PolicyPal, an insurance policy assistant.
Always answer using the retrieved context below.
If the context contains lists or bullet points, you MUST format them as a valid Markdown list.
CRITICAL FORMATTING RULE FOR INLINE BULLETS:
If you encounter a line in the context containing inline bullet points separated horizontally (e.g., "• Item A • Item B • Item C"), you MUST split them apart. Convert every single bullet character into a brand-new vertical list item on its own separate line using Markdown formatting.
Ensure there is a newline before starting the list, and separate each bullet point item with a double newline (\n\n) so they render on separate vertical lines.
Do not compress them into a single sentence.
Format lists with bullet points (•) or numbers for clarity.
Do not summarize with placeholders like [Insert details].
If the context does not contain the answer, say 'Not mentioned in the document.', don't fabricate an answer.


Context:
{context}
Question:
{question}
Answer:
"""
)


db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4,
                   "fetch_k": 20,
                   "lambda_mult": 0.4
                   }
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template}
)

if "messages" not in st.session_state:
    st.session_state["messages"] = []
if "chat_input" not in st.session_state:
    st.session_state["chat_input"] = ""
if "last_query" not in st.session_state:
    st.session_state["last_query"] = ""

answer = " "
def handle_submit():
    user_text = st.session_state["chat_input"].strip()
    if user_text:

        st.session_state["last_query"] = user_text

        response = qa.invoke(user_text)
        answer = response["result"]
        st.session_state["messages"].append(("You", user_text))
        st.session_state["messages"].append(("PolicyPal", answer))
        st.session_state["chat_input"] = ""

st.title("Policy & Claims Copilot")
st.subheader("Welcome! You are chatting with PolicyPal")
user_query = st.text_input("Enter your claim query:", key="chat_input", on_change=handle_submit)


query_to_check = st.session_state.get("last_query", "").lower().strip()
pre_checks = []
if "hospitalization" in query_to_check:
    pre_checks.append("Hospitalization must be ≥ 24 hours.")
if "maternity" in query_to_check:
    pre_checks.append("Maternity covered only after 2 years waiting period, limit ₹50,000.")
if "dental" in query_to_check:
    pre_checks.append("Dental treatment is excluded.")


messages = st.session_state.get("messages", [])

if messages:
    current = messages[-2:]
    for sender, msg in current:
        if sender == "You":
            st.markdown(f"**You:** {msg}")
        elif pre_checks:
            st.markdown("#Pre-check Validation")
            for rule in pre_checks:
                st.write("- " + rule)
        else:
            st.markdown(f"**PolicyPal says:** {msg}")

